# **DeconvNet: carrying a channel's response back to the pixels**

Practice for the module [“Analysis of CNN-based Models”](https://ai-interpretability.school), lesson “DeconvNet: what lit up the channel in this picture”.

In the lesson, DeconvNet ([Zeiler and Fergus, 2013](https://arxiv.org/abs/1311.2901)) is described this way: take the response of one channel at one position, zero everything else and carry the remaining number down through the network, inverting each operation: a convolution by a transposed convolution, pooling through the remembered positions of the maxima, ReLU by rectification. Here we build this by hand, check it against `captum.attr.Deconvolution`, and test how far such a picture can be trusted:

1. AlexNet and the photo we will look at;
2. DeconvNet by hand: one rule for ReLU on the backward pass;
3. the Captum library call and a comparison;
4. three ReLU rules side by side: the gradient, DeconvNet, guided backpropagation;
5. different layers and channels;
6. where it breaks: the picture barely depends on the class or on the weights of the top layers.

AlexNet is the torchvision architecture closest to the network of Zeiler and Fergus. The notebook runs on CPU; the whole run takes about a minute, and the AlexNet weights (about 230 MB) are downloaded on the first run.

Cat and pug photo: [Priscilla Du Preez, Unsplash](https://unsplash.com/photos/qtkpOxCpl0Y), Unsplash License.

In [ ]:
!pip install -q captum==0.9.0

In [ ]:
import io
import copy
import urllib.request
import warnings
warnings.filterwarnings('ignore', message='IProgress not found')   # tqdm without ipywidgets; Colab has ipywidgets
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import models, transforms
from captum.attr import Deconvolution, GuidedBackprop, Saliency, NeuronDeconvolution

warnings.filterwarnings('ignore', category=UserWarning, module='captum')   # Captum warns that it enables the input gradient and sets hooks itself
torch.manual_seed(0)

model = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1).eval()
for m in model.modules():
    if isinstance(m, torch.nn.ReLU):
        m.inplace = False            # backward hooks need tensors that were not overwritten
print(model.features)

## 1. The photo

A photo with a cat and a dog from the `data` folder of the course repository. Standard ImageNet preprocessing: side 256, a 224 center crop, normalization by the ImageNet channel means and spreads.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
prep = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
                           transforms.Normalize(MEAN, STD)])
raw = urllib.request.urlopen('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/cat_and_dog.jpg', timeout=30).read()
x = prep(Image.open(io.BytesIO(raw)).convert('RGB')).unsqueeze(0)
labels = models.AlexNet_Weights.IMAGENET1K_V1.meta['categories']

with torch.no_grad():
    probs = model(x).softmax(1)[0]
top5 = probs.topk(5)
for p, i in zip(top5.values, top5.indices):
    print(f'{p:.3f}  {i.item():4d}  {labels[i]}')


def to_img(t):
    # input tensor (1, 3, H, W) → image in [0, 1] for imshow
    return (t[0].permute(1, 2, 0) * torch.tensor(STD) + torch.tensor(MEAN)).clamp(0, 1).numpy()


def to_map(a):
    # attribution (1, 3, H, W) → image: each channel scaled by its own largest absolute value
    a = a[0].detach().permute(1, 2, 0).numpy()
    return np.clip(0.5 + a / (2 * np.abs(a).max() + 1e-12), 0, 1)


def corr(a, b):
    return np.corrcoef(a.detach().flatten().numpy(), b.detach().flatten().numpy())[0, 1]

## 2. DeconvNet by hand

How each operation is inverted:

- a **convolution** is inverted by a transposed convolution with the same filters, which is exactly what the autograd backward pass through `Conv2d` does;
- **max pooling** is inverted through the remembered positions of the maxima (switches): the value returns to where the maximum came from, the other positions get zero, which is also the ordinary backward pass through `MaxPool2d`;
- **ReLU** is the only place where DeconvNet differs from the gradient. The gradient passes the signal where the ReLU input was positive **on the forward pass**. DeconvNet looks not at the forward pass but at the backward signal itself and passes its positive part: $R_{\text{in}}=\max(R_{\text{out}},0)$.

So the whole DeconvNet is an ordinary backward pass in which every ReLU has its rule replaced. We replace it with a `register_full_backward_hook`: the hook receives the gradient with respect to the module output and returns what goes further down.

The target is a channel of the last convolutional block `features[11]` (the ReLU after the fifth convolution), at the position where this channel responds most strongly. We pick the channel with the largest response on the photo.

In [ ]:
LAYER = model.features[11]
captured = {}
hook = LAYER.register_forward_hook(lambda m, i, o: captured.__setitem__('a', o))
with torch.no_grad():
    model(x)
hook.remove()
A = captured['a'][0]                                     # (256, 13, 13)
CHANNEL = A.amax(dim=(1, 2)).argmax().item()
POS = tuple(int(p) for p in np.unravel_index(A[CHANNEL].argmax().item(), A[CHANNEL].shape))
print(f'channel {CHANNEL}, position {POS} in a {tuple(A.shape[1:])} map, response {A[CHANNEL][POS]:.2f}')

**Task 1. Write a function `backward_with_rule(rule, layer, channel, pos)`. It attaches to every `nn.ReLU` of `model` a backward hook with rule `rule` (a function of the gradient with respect to the output and the saved ReLU input), runs the forward pass up to the layer `layer`, takes the response of `channel` at position `pos` (everything else is zeroed automatically: we differentiate a single number), runs the backward pass to the input and returns the gradient with respect to the input, removing the hooks. The DeconvNet rule is `lambda g, inp: g.clamp(min=0)`. Compute `deconv_hand` for channel `CHANNEL` at position `POS` and plot the photo next to the picture.**

Hint: the ReLU input of the forward pass can be stored with a forward hook of the same module; DeconvNet does not need it, but it will be useful in section 4.

In [ ]:
# Your code here

## 3. The library call

In Captum, DeconvNet for the model output is `captum.attr.Deconvolution(model).attribute(x, target=class)`, for an internal neuron `NeuronDeconvolution(model, layer).attribute(x, neuron_selector=(channel, row, column))`. Captum attaches hooks to `nn.ReLU` modules, so functional calls `F.relu` inside `forward` are invisible to it, and a ReLU module used twice gets the rule twice; in AlexNet every ReLU is a separate module.

In [ ]:
deconv_captum = NeuronDeconvolution(model, LAYER).attribute(x, neuron_selector=(CHANNEL,) + POS)
print(f'correlation of hand DeconvNet and NeuronDeconvolution: {corr(deconv_hand, deconv_captum):.4f}')
print(f'largest discrepancy: {(deconv_hand - deconv_captum).abs().max():.2e}')

## 4. Three rules for ReLU

A backward-pass visualization method is defined by what ReLU does to the signal:

| rule | what ReLU passes | formula |
|---|---|---|
| gradient (vanilla gradient) | where the ReLU input on the forward pass is positive | $g\cdot[z>0]$ |
| DeconvNet | the positive part of the backward signal | $\max(g,0)$ |
| guided backpropagation ([Springenberg et al., 2015](https://arxiv.org/abs/1412.6806)) | both conditions at once | $\max(g,0)\cdot[z>0]$ |

**Task 2. Use `backward_with_rule` to compute the gradient and guided backpropagation pictures for the same channel and position and plot the three pictures side by side. Print the pairwise correlations of the three pictures.**

In [ ]:
# Your code here

## 5. Different layers

Zeiler and Fergus showed that the higher the layer, the larger and more complex the structure DeconvNet recovers: edges and color blobs in the first layers, object parts in the last. We take the strongest channel at the output of each of the five convolutional blocks (ReLU `features[1]`, `[4]`, `[7]`, `[9]`, `[11]`) and for each compute DeconvNet from its maximal position. The box on the photo shows the region where the DeconvNet picture is nonzero, the effective receptive field of this response.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7.5))
for k, idx in enumerate((1, 4, 7, 9, 11)):
    layer = model.features[idx]
    hook = layer.register_forward_hook(lambda m, i, o: captured.__setitem__('a', o))
    with torch.no_grad():
        model(x)
    hook.remove()
    Al = captured['a'][0]
    ch = Al.amax(dim=(1, 2)).argmax().item()
    pos = tuple(int(p) for p in np.unravel_index(Al[ch].argmax().item(), Al[ch].shape))
    a = backward_with_rule(lambda g, z: g.clamp(min=0), layer, ch, pos)
    nz = np.argwhere(a[0].abs().sum(0).numpy() > 0)
    (r0, c0), (r1, c1) = nz.min(0), nz.max(0)
    axes[0, k].imshow(to_img(x))
    axes[0, k].add_patch(plt.Rectangle((c0, r0), c1 - c0, r1 - r0, fill=False, color='#c0392b', lw=2))
    axes[0, k].set_title(f'features[{idx}], channel {ch}\nnonzero region {r1 - r0 + 1}×{c1 - c0 + 1} px')
    axes[1, k].imshow(to_map(a)[r0:r1 + 1, c0:c1 + 1])
    axes[1, k].set_title('DeconvNet, cropped to the region')
for ax in axes.flat:
    ax.axis('off')
fig.tight_layout()
plt.show()

## 6. Where it breaks

The DeconvNet picture is pretty and “looks like the object”. We check whether it answers the questions we ask of it. Two checks:

- **class sensitivity.** Compute `Deconvolution` for the most probable class (`pug`, a dog) and for the class `tabby` (a striped cat); the photo contains both a dog and a cat. If the explanation is about the class, the pictures for the two classes must differ;
- **weight sensitivity** ([Adebayo et al., 2018](https://arxiv.org/abs/1810.03292)). Replace the weights of the classifier and of the last convolution with random ones. If the explanation is about what the network learned, the picture must change.

For comparison, the same checks for the gradient (`Saliency`) and guided backpropagation (`GuidedBackprop`).

**Task 3. For each of the three methods (`Saliency` with `abs=False`, `Deconvolution`, `GuidedBackprop`) compute the correlation of the pictures for classes `c1 = top5.indices[0]` and `c2 = labels.index('tabby')`, and the correlation of the picture for class `top5.indices[0]` of the original model and of the model `random_model`, whose `classifier` and `features[10]` weights are replaced by normal noise with standard deviation 0.01 (`torch.nn.init.normal_`, after `torch.manual_seed(0)`). Build a table.**

In [ ]:
# Your code here

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for row, (name, method) in enumerate((('Gradient', Saliency), ('DeconvNet', Deconvolution), ('Guided backprop', GuidedBackprop))):
    for col, (title, net, tgt) in enumerate(((f'class {labels[c1]}', model, c1), (f'class {labels[c2]}', model, c2),
                                             (f'class {labels[c1]}, random top layers', random_model, c1))):
        axes[row, col].imshow(to_map(attribution(method, net, tgt)))
        axes[row, col].set_title(f'{name}\n{title}', fontsize=9)
        axes[row, col].axis('off')
fig.tight_layout()
plt.show()

**Task 4. From the table of task 3: which of the three methods passed both checks, and what does this say about where the DeconvNet picture comes from?**

**Your answer:**

## What to take away

- **DeconvNet is a backward pass with a different ReLU rule.** Ordinary autograd inverts convolution and pooling, and the whole difference from the gradient is that ReLU passes the positive part of the backward signal.
- **The hand version matches `captum.attr.NeuronDeconvolution`.** Captum attaches the same hooks to `nn.ReLU` modules; a functional `F.relu` is invisible to it.
- **The higher the layer, the larger the region of the photo behind a response.** This is the receptive field of the response, and DeconvNet shows which pixels inside it took part.
- **The DeconvNet picture barely depends on the class or on the weights of the top layers.** It shows well what the lower layers look at in the photo, and poorly why the network made its decision. The question about the decision needs methods that pass the weight randomization check.

Sources:
1. [Zeiler & Fergus, 2013. Visualizing and understanding convolutional networks](https://arxiv.org/abs/1311.2901)
2. [Springenberg et al., 2015. Striving for simplicity: the all convolutional net](https://arxiv.org/abs/1412.6806)
3. [Adebayo et al., 2018. Sanity checks for saliency maps](https://arxiv.org/abs/1810.03292)
4. [Nie, Zhang & Patel, 2018. A theoretical explanation for perplexing behaviors of backpropagation-based visualizations](https://arxiv.org/abs/1805.07039)
5. [Captum: Deconvolution](https://captum.ai/api/deconvolution.html)